[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C54_DETR_Set_Prediction_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与两种范式的输出差异（NMS / 一对一匹配 / 集合损失）

目标：用一个**合成的 TSR 场景**把「密集预测 + NMS」和「一对一集合预测」两种范式
放在同一组数据上跑一遍，亲眼看到它们的输出差在哪、代价差在哪。

本 notebook 你会亲手实现：
1. **IoU 矩阵与 NMS**，并测出「同一场景里 NMS 阈值从 0.59 变到 0.60，输出框数从 5 跳到 6」
2. **NMS 阈值的可行窗口**，并在一个真实 TSR 场景（组合标志牌）上证明**这个窗口是空的**
3. **一对一集合预测的输出**：不做任何抑制，重复率天然为 0
4. **最小版的三机制玩具**：代价矩阵 → 暴力最优匹配 → 集合损失
5. **后处理代价对比**：NMS 的 O(n²) vs 集合预测的 O(N)

> 心智模型：**NMS 是在推理期删掉重复，一对一匹配是在训练期不让重复长出来。**

## 1 · 环境自检

In [ ]:
import sys, platform, json, math, itertools
import numpy as np

print('Python', sys.version.split()[0], '|', platform.system(), platform.machine())
print('numpy ', np.__version__)
for name in ['torch', 'scipy', 'mmdet']:
    try:
        mod = __import__(name)
        print(f'  {name:<8s} {getattr(mod, "__version__", "?")}  (本课**不使用**它)')
    except ImportError:
        print(f'  {name:<8s} 未安装  ->  本课本来就不需要')

rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)
print('\n✅ 环境就绪：纯 numpy + 标准库，CPU，不联网。')
print('⚠️  本课**禁止** scipy.optimize.linear_sum_assignment —— 匈牙利算法要在模块 01 手写。')

## 2 · 合成一个 TSR 场景：一块牌，几个框？

1920×1080 的车载前视图像，三块交通标志：一块远处的小限速牌（24×24 px）、
一块中距禁令牌（40×40 px）、一块近处警告牌（60×60 px）。

**密集预测器**对每块牌都会吐出好几个高分框（因为周围好几个格子/anchor 都判定「这里有目标」），
外加两个背景误检（路边广告牌、前车车身贴纸）——这两类是 TSR 里最经典的假正例来源。

In [ ]:
# GT：xyxy 像素坐标
GT = np.array([
    [ 900., 300.,  924.,  324.],   # 0  远处限速60      24x24 px
    [1200., 380., 1240.,  420.],   # 1  中距禁止超车    40x40 px
    [ 640., 420.,  700.,  480.],   # 2  近处注意行人    60x60 px
])
GT_NAME = ['限速60(远,24px)', '禁止超车(中,40px)', '注意行人(近,60px)']

def shift(box, dx, dy):
    return box + np.array([dx, dy, dx, dy], dtype=float)

# 每块牌配几个「重复框」：位移几个像素、分数递减 —— 这就是密集预测器的真实行为
_SPEC = [
    ([0, 3, -3, 2], [0, 0, 2, -3], [0.92, 0.85, 0.78, 0.71]),   # GT0 的 4 个框
    ([0, 5, -4, 6], [0, 0, 4, -5], [0.88, 0.80, 0.74, 0.69]),   # GT1 的 4 个框
    ([0, 8, -6],    [0, 0, 6],     [0.95, 0.83, 0.72]),         # GT2 的 3 个框
]
_rows = []
for k, (dxs, dys, scs) in enumerate(_SPEC):
    for dx, dy, s in zip(dxs, dys, scs):
        _rows.append((shift(GT[k], dx, dy), s, k))
_rows.append((np.array([ 300., 600.,  340., 640.]), 0.61, -1))   # 背景误检：路边广告牌
_rows.append((np.array([1500., 200., 1530., 230.]), 0.55, -1))   # 背景误检：前车车身贴纸

P_BOX   = np.array([r[0] for r in _rows])
P_SCORE = np.array([r[1] for r in _rows])
P_SRC   = np.array([r[2] for r in _rows])       # 这个框来自哪个 GT，-1 表示背景误检

print(f'GT 数量        : {len(GT)}')
print(f'密集预测框数量 : {len(P_BOX)}   (其中背景误检 {int((P_SRC == -1).sum())} 个)')
print(f'{"idx":>3s} {"来源":>16s} {"score":>6s}  box')
for i in range(len(P_BOX)):
    src = GT_NAME[P_SRC[i]] if P_SRC[i] >= 0 else '背景误检'
    print(f'{i:>3d} {src:>16s} {P_SCORE[i]:>6.2f}  {P_BOX[i]}')

assert len(P_BOX) == 13 and (P_SRC == -1).sum() == 2
print('\n⚠️  3 块牌 -> 13 个高分框。**模型自己不去重**，因为每个位置都是独立判断的。')

## 3 · 范式 A：密集预测 + NMS

NMS 的规则只有一句：**按分数从高到低，保留当前最高分的框，删掉所有与它 IoU 超过阈值的框。**

先实现 IoU 矩阵和 NMS，然后扫一遍阈值，看输出框数怎么变。

In [ ]:
def iou_matrix(a, b):
    """a:(N,4) b:(M,4) 均为 xyxy -> (N,M) 的 IoU 矩阵。"""
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-12)

def nms(boxes, scores, iou_thr, score_thr=0.5):
    """贪心 NMS：返回保留下来的框下标（按分数降序）。"""
    idx = np.where(scores >= score_thr)[0]
    idx = idx[np.argsort(-scores[idx])]
    keep = []
    while len(idx):
        i = idx[0]
        keep.append(int(i))
        if len(idx) == 1:
            break
        ious = iou_matrix(boxes[i:i + 1], boxes[idx[1:]])[0]
        idx = idx[1:][ious <= iou_thr]          # IoU 超过阈值的被抑制
    return keep

# 自校验：一个框和自己的 IoU 必须是 1
assert abs(iou_matrix(P_BOX[:1], P_BOX[:1])[0, 0] - 1.0) < 1e-9
# 手算校验：GT0 的框 [900,300,924,324] 与它右移 3px 的副本
#   交集 21*24=504，并集 576*2-504=648  ->  IoU = 504/648 = 0.7778
assert abs(iou_matrix(P_BOX[0:1], P_BOX[1:2])[0, 0] - 504 / 648) < 1e-9
print('IoU 实现通过手算校验 ✅  (0.7778 = 504/648)')

print(f'\n{"IoU 阈值":>10s} {"保留框数":>10s}   说明')
for t in [0.10, 0.30, 0.50, 0.59, 0.60, 0.70, 0.90]:
    k = nms(P_BOX, P_SCORE, t)
    note = '重复框全被删掉（3 真 + 2 误检）' if len(k) == 5 else \
           ('几乎不抑制，13 个框全留' if len(k) == 13 else '开始漏删重复框')
    print(f'{t:>10.2f} {len(k):>10d}   {note}')

assert len(nms(P_BOX, P_SCORE, 0.50)) == 5
assert len(nms(P_BOX, P_SCORE, 0.70)) == 10
assert len(nms(P_BOX, P_SCORE, 0.90)) == 13
print('\n⚠️  阈值从 0.59 到 0.60，输出就从 5 个变成 6 个 —— **NMS 对阈值是硬切换，没有过渡带**。')

In [ ]:
# 「每个 GT 被几个框认领」—— 这就是重复率
def duplicates_per_gt_ref(pred_boxes, pred_scores, gt_boxes, score_thr=0.5, iou_thr=0.5):
    M = iou_matrix(pred_boxes, gt_boxes)
    return ((pred_scores[:, None] >= score_thr) & (M >= iou_thr)).sum(axis=0)

before = duplicates_per_gt_ref(P_BOX, P_SCORE, GT)
keep = nms(P_BOX, P_SCORE, 0.50)
after = duplicates_per_gt_ref(P_BOX[keep], P_SCORE[keep], GT)

print(f'{"GT":>18s} {"NMS 前":>8s} {"NMS 后":>8s}')
for j in range(len(GT)):
    print(f'{GT_NAME[j]:>18s} {before[j]:>8d} {after[j]:>8d}')
assert list(before) == [4, 4, 3] and list(after) == [1, 1, 1]
print('\n✅ NMS 确实把重复率从 [4,4,3] 压到 [1,1,1] —— 它是有效的。')
print('   问题不在「有没有效」，而在「它是一条写死的全局规则」。下一节量化这一点。')

## 4 · NMS 的阈值困境：一个真实的 TSR 反例

NMS 的 IoU 阈值必须同时满足两个互相冲突的要求：

- **要删掉重复框** → 阈值必须 **低于**「同一目标的重复框之间的 IoU」
- **要保住重叠的真目标** → 阈值必须 **不低于**「两个真目标之间的 IoU」

这两个条件在同一个数据集上未必有交集。下面用一个 TSR 里天天见的场景来证明：
**组合标志牌**——一块大的蓝底指路牌里嵌着一块限速牌，两个都是**必须检出的真目标**。

In [ ]:
# 重复框与「它所属 GT 的最高分框」之间的 IoU
top_of = [int(np.argmax(np.where(P_SRC == k, P_SCORE, -1.0))) for k in range(len(GT))]
dup_ious = []
for k in range(len(GT)):
    for i in np.where(P_SRC == k)[0]:
        if i != top_of[k]:
            dup_ious.append(float(iou_matrix(P_BOX[top_of[k]:top_of[k]+1], P_BOX[i:i+1])[0, 0]))
dup_ious = np.array(dup_ious)
print('重复框对的 IoU :', np.round(dup_ious, 4))
print(f'  -> 要删光重复框，阈值必须 < {dup_ious.min():.4f}')

# 组合标志牌：大指路牌 120x120，里面嵌一块 100x100 的限速牌（**两个都是真目标**）
OUTER = np.array([[980., 560., 1100., 680.]])    # 蓝底指路牌
INNER = np.array([[990., 570., 1090., 670.]])    # 嵌在里面的限速牌
iou_pair = float(iou_matrix(OUTER, INNER)[0, 0])
print(f'\n组合标志牌：外牌与内牌的 IoU = {iou_pair:.4f}  (= 10000/14400，内牌完全被外牌包住)')
print(f'  -> 要同时保住这两个真目标，阈值必须 >= {iou_pair:.4f}')

lo, hi = iou_pair, float(dup_ious.min())
print(f'\n可行阈值窗口 = [{lo:.4f}, {hi:.4f})   ->  {"非空" if lo < hi else "**空集**"}')
assert lo > hi, '这个场景下窗口应当是空的'
print('❌ 没有任何一个 IoU 阈值能同时做到「删光重复」和「不误删真目标」。')

# 实测：用能删光重复的阈值 0.5 去跑组合牌，会发生什么
pair_box = np.vstack([OUTER, INNER]); pair_score = np.array([0.90, 0.85])
k05 = nms(pair_box, pair_score, 0.50)
k70 = nms(pair_box, pair_score, 0.70)
print(f'\n组合牌在 iou_thr=0.50 下保留 {len(k05)} 个框  -> **限速牌被当成重复删掉了**')
print(f'组合牌在 iou_thr=0.70 下保留 {len(k70)} 个框  -> 保住了，但此时主场景的重复框也全留下')
assert len(k05) == 1 and len(k70) == 2

In [ ]:
# 有人会说：用 class-wise NMS（只在同类之间抑制）不就行了？
# —— 它能救「外牌+内牌」（类别不同），但救不了「同类的相邻标志」。
GANTRY = np.array([          # 龙门架上两块**同类**限速牌，检测器把它们框得一大一小
    [700., 300., 820., 380.],       # 检成一个把两块都圈进去的大框
    [700., 300., 790., 380.],       # 只圈住左边那块
])
g_score = np.array([0.88, 0.84])
print(f'同类相邻标志的 IoU = {iou_matrix(GANTRY[:1], GANTRY[1:])[0,0]:.4f}')
print(f'class-wise NMS(0.5) 后保留 {len(nms(GANTRY, g_score, 0.5))} 个 —— 同类之间照样抑制')
assert len(nms(GANTRY, g_score, 0.5)) == 1

print('''
✅ 本节结论（面试高频）：
   NMS 的失败**不是实现问题，是范式问题** —— 它用一个全局标量阈值去回答
   「这两个框是不是同一个目标」，而这个问题的答案**依赖场景**。
   class-wise NMS / soft-NMS / WBF 都只是在缓解，没有改变「用规则代替学习」这一点。
   一对一匹配的做法是：**根本不问这个问题**，直接让模型学会一个目标只输出一个框。''')

## 5 · 范式 B：一对一集合预测

集合预测器输出**恰好 N 个**槽位（object query），每个槽位一个 (score, box)。
因为训练时用的是一对一匹配，训练收敛后**不会有两个 query 认领同一个目标**。

这里我们不训练模型（那是模块 03–04 的事），直接模拟一个训练好的 DETR 的输出：
N=20 个 query，其中 3 个精准命中 3 块牌，2 个是背景误检，其余 15 个是 no-object。

In [ ]:
N_QUERY = 20
q_box = np.zeros((N_QUERY, 4)); q_score = np.zeros(N_QUERY)

# 3 个 query 各认领一块牌（框有 1-2 px 的正常回归误差）
q_box[3]  = shift(GT[0],  1., -1.); q_score[3]  = 0.93
q_box[11] = shift(GT[1], -2.,  1.); q_score[11] = 0.90
q_box[7]  = shift(GT[2],  1.,  2.); q_score[7]  = 0.96
# 2 个背景误检（任何检测器都会有）
q_box[15] = np.array([ 300., 600.,  340., 640.]); q_score[15] = 0.61
q_box[18] = np.array([1500., 200., 1530., 230.]); q_score[18] = 0.55
# 其余 15 个 query 是 no-object：分数极低，框是「没收敛的乱框」
idle = [i for i in range(N_QUERY) if q_score[i] == 0]
q_score[idle] = rng.uniform(0.01, 0.08, size=len(idle))
q_box[idle]   = np.column_stack([rng.uniform(0, 1700, len(idle)), rng.uniform(0, 900, len(idle)),
                                 np.zeros(len(idle)), np.zeros(len(idle))])
q_box[idle, 2] = q_box[idle, 0] + rng.uniform(20, 200, len(idle))
q_box[idle, 3] = q_box[idle, 1] + rng.uniform(20, 200, len(idle))

# 集合预测的「后处理」：只有一个分数阈值，**没有 NMS**
out = np.where(q_score >= 0.5)[0]
print(f'N = {N_QUERY} 个 query，score>=0.5 的输出 {len(out)} 个：{list(out)}')
dup = duplicates_per_gt_ref(q_box[out], q_score[out], GT)
print(f'每个 GT 被几个输出认领: {dup.tolist()}   (没有做任何抑制)')
assert len(out) == 5 and list(dup) == [1, 1, 1]
print('✅ 重复率天然为 1 —— 因为训练时的一对一匹配已经把重复压制掉了。')

# 同一个组合标志牌场景：集合预测不做抑制，两块牌都留下
pair_q_score = np.array([0.90, 0.85])
pair_out = np.where(pair_q_score >= 0.5)[0]
print(f'\n组合标志牌：集合预测输出 {len(pair_out)} 个框（外牌+内牌都保住）')
assert len(pair_out) == 2
print('✅ **没有抑制步骤，就不可能误删真目标** —— 这是 TSR 组合牌场景的关键收益。')

## 6 · 三个机制的最小玩具：query / 二分匹配 / 集合损失

现在把训练那一侧也走一遍。用归一化的 `cxcywh` 坐标（DETR 的约定），
N=6 个 query、M=2 个 GT，走完整条链路：

**代价矩阵 → 一对一最优匹配（暴力枚举，模块 01 会换成 O(n³) 的匈牙利）→ 集合损失**

In [ ]:
def cxcywh_to_xyxy(b):
    cx, cy, w, h = b[..., 0], b[..., 1], b[..., 2], b[..., 3]
    return np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], axis=-1)

def giou_matrix(a_xyxy, b_xyxy):
    """GIoU = IoU - (C - union)/C，C 是最小外接框面积。范围 [-1, 1]。"""
    a, b = a_xyxy, b_xyxy
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); bb = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    union = aa[:, None] + bb[None, :] - inter
    iou = inter / (union + 1e-12)
    ex1 = np.minimum(a[:, None, 0], b[None, :, 0]); ey1 = np.minimum(a[:, None, 1], b[None, :, 1])
    ex2 = np.maximum(a[:, None, 2], b[None, :, 2]); ey2 = np.maximum(a[:, None, 3], b[None, :, 3])
    C = np.clip(ex2 - ex1, 0, None) * np.clip(ey2 - ey1, 0, None)
    return iou - (C - union) / (C + 1e-12)

# 6 个 query 的输出（归一化 cxcywh）+ 每个 query 对目标类的概率
q_boxes = np.array([[0.30, 0.50, 0.05, 0.05],   # q0 几乎压在 GT0 上
                    [0.32, 0.52, 0.06, 0.06],   # q1 也很靠近 GT0 —— **重复框的来源**
                    [0.70, 0.40, 0.09, 0.09],   # q2 靠近 GT1
                    [0.10, 0.20, 0.20, 0.20],   # q3 空
                    [0.85, 0.80, 0.15, 0.15],   # q4 空
                    [0.50, 0.50, 0.40, 0.40]])  # q5 一个大而无当的框
q_prob  = np.array([0.80, 0.65, 0.75, 0.05, 0.04, 0.10])
g_boxes = np.array([[0.30, 0.50, 0.04, 0.04],   # GT0 远处小限速牌
                    [0.70, 0.40, 0.08, 0.08]])  # GT1 中距禁令牌

W_CLS, W_L1, W_GIOU = 1.0, 5.0, 2.0
l1   = np.abs(q_boxes[:, None, :] - g_boxes[None, :, :]).sum(-1)
gi   = giou_matrix(cxcywh_to_xyxy(q_boxes), cxcywh_to_xyxy(g_boxes))
COST = W_CLS * (-q_prob[:, None]) + W_L1 * l1 + W_GIOU * (-gi)

print('代价矩阵 C[query, GT]   (越小越该配对)')
print('        ' + '  '.join(f'GT{j}' + ' ' * 4 for j in range(len(g_boxes))))
for i in range(len(q_boxes)):
    print(f'  q{i}  ' + '  '.join(f'{COST[i, j]:7.3f}' for j in range(len(g_boxes))))
assert COST.shape == (6, 2)

In [ ]:
def brute_force_assignment(cost):
    """暴力枚举所有一对一指派，返回 (row_ind, col_ind, 最小总代价)。
       cost:(N,M)。N>=M 时枚举「哪 M 个 query 去认领这 M 个 GT」。"""
    a = np.asarray(cost, dtype=float); n, m = a.shape
    best, best_pair = np.inf, None
    if n <= m:
        for perm in itertools.permutations(range(m), n):
            t = float(sum(a[i, perm[i]] for i in range(n)))
            if t < best:
                best, best_pair = t, (np.arange(n), np.array(perm))
    else:
        for perm in itertools.permutations(range(n), m):
            t = float(sum(a[perm[j], j] for j in range(m)))
            if t < best:
                r = np.array(perm); c = np.arange(m); o = np.argsort(r)
                best, best_pair = t, (r[o], c[o])
    return best_pair[0], best_pair[1], best

rows, cols, total = brute_force_assignment(COST)
print(f'枚举了 C(6,2)*2! = {6*5} 种指派')
for r, c in zip(rows, cols):
    print(f'  GT{c} 由 q{r} 认领   代价 {COST[r, c]:.3f}')
print(f'总代价 = {total:.3f}')
assert list(rows) == [0, 2] and list(cols) == [0, 1]
print('\n⚠️  注意 q1 也很靠近 GT0（IoU 不低、分数 0.65），但它**没有**被匹配上。')
print('    下一段就是 DETR 全部魔力的所在：没匹配上的 query 会被判为 no-object。')

In [ ]:
def set_loss(cost_rows, cost_cols, q_prob, q_boxes, g_boxes, w_noobj=0.1):
    """集合损失：匹配上的算「分类为前景 + 框」，没匹配上的算「分类为 no-object」。"""
    matched = set(int(r) for r in cost_rows)
    l_cls_pos = l_box = l_cls_neg = 0.0
    for r, c in zip(cost_rows, cost_cols):
        l_cls_pos += -np.log(q_prob[r] + 1e-12)
        l1_ = np.abs(q_boxes[r] - g_boxes[c]).sum()
        gi_ = giou_matrix(cxcywh_to_xyxy(q_boxes[r:r+1]), cxcywh_to_xyxy(g_boxes[c:c+1]))[0, 0]
        l_box += W_L1 * l1_ + W_GIOU * (1.0 - gi_)
    for i in range(len(q_prob)):
        if i not in matched:
            l_cls_neg += -w_noobj * np.log(1.0 - q_prob[i] + 1e-12)
    return l_cls_pos, l_box, l_cls_neg

lp, lb, ln = set_loss(rows, cols, q_prob, q_boxes, g_boxes)
print(f'匹配上的分类损失 (推向前景)   = {lp:.4f}   作用于 q{list(rows)}')
print(f'匹配上的框损失   (推向 GT)     = {lb:.4f}')
print(f'未匹配的分类损失 (推向 no-obj) = {ln:.4f}   作用于 q{[i for i in range(6) if i not in set(rows)]}')
print(f'总损失 = {lp + lb + ln:.4f}')
assert lp > 0 and lb > 0 and ln > 0

# 关键实验：把 q1（那个「重复框」）的分数调高，看它承受多大的压制力
for p1 in [0.20, 0.65, 0.90]:
    qp = q_prob.copy(); qp[1] = p1
    C2 = W_CLS * (-qp[:, None]) + W_L1 * l1 + W_GIOU * (-gi)
    r2, c2, _ = brute_force_assignment(C2)
    _, _, ln2 = set_loss(r2, c2, qp, q_boxes, g_boxes)
    fired = 1 in set(int(x) for x in r2)
    print(f'  q1 的前景概率 = {p1:.2f} -> 被匹配? {fired} | 未匹配项损失 = {ln2:.4f}')
print('''
✅ **q1 越自信，它作为「未匹配 query」承受的 no-object 损失就越大。**
   这就是「训练期压制重复」的全部机制：一个目标只有一个 query 能被判为前景，
   其余想认领它的 query 都会被 no-object 损失按下去。
   而 NMS 是在推理期把它们**删掉**（模型自己还是学不会）。''')

## 7 · 后处理代价：NMS 随目标数增长，集合预测不随

这一节量化的是**车端最关心的那件事**：延迟的确定性。

- NMS 是贪心迭代，最坏情况要做 `n(n-1)/2` 次 IoU 比较，**n 是通过分数阈值的候选框数**
- 集合预测的后处理是 `N` 次分数比较，**N 是固定的 query 数**，与场景内容无关

In [ ]:
def nms_comparisons(n):        return n * (n - 1) // 2      # 最坏情况的两两 IoU 比较次数
def set_comparisons(n_query):  return n_query                 # N 次分数阈值比较

print(f'{"候选框数 n":>12s} {"NMS 比较次数":>14s} {"集合预测(N=300)":>16s} {"倍数":>8s}')
for n in [10, 50, 100, 300, 1000, 3000]:
    a, b = nms_comparisons(n), set_comparisons(300)
    print(f'{n:>12d} {a:>14d} {b:>16d} {a / b:>8.1f}x')

assert nms_comparisons(1000) == 499500
assert nms_comparisons(1000) / set_comparisons(1000) > 400

# 场景内容如何影响延迟：市区路口 vs 高速空旷路段
scenes = [('高速空旷（1 块牌）', 12), ('城市普通路段（3 块牌）', 40),
          ('复杂路口（12 块牌 + 广告牌）', 260), ('隧道口逆光（大量低分误检）', 900)]
print(f'\n{"场景":>28s} {"候选框":>8s} {"NMS 比较":>10s} {"集合预测":>10s}')
costs_nms = []
for name, n in scenes:
    costs_nms.append(nms_comparisons(n))
    print(f'{name:>28s} {n:>8d} {nms_comparisons(n):>10d} {set_comparisons(300):>10d}')
spread = max(costs_nms) / min(costs_nms)
print(f'\nNMS 后处理开销在不同场景之间相差 {spread:.0f} 倍；集合预测相差 1.0 倍。')
assert spread > 100
print('''
✅ 面试标准答案：**DETR 系的价值不是「平均延迟更低」（通常更高），
   而是「延迟方差更小、p99 更接近 p50」**，因为它没有随目标数变化的后处理。
   车端功能安全关心的是**尾延迟**：一帧超时就丢一帧感知结果。''')

## ✏️ 练习 1：NMS 阈值的可行窗口

实现 `feasible_nms_window(dup_ious, keep_ious)`：

- `dup_ious`：所有「同一目标的重复框对」的 IoU 列表 → 要删光它们，阈值必须 **严格小于** 其最小值
- `keep_ious`：所有「必须同时保留的真目标对」的 IoU 列表 → 要保住它们，阈值必须 **大于等于** 其最大值

返回 `(lo, hi, ok)`：窗口为 `[lo, hi)`，`ok` 表示窗口是否非空（`lo < hi`）。
两个列表可能为空：`keep_ious` 为空时 `lo = 0.0`；`dup_ious` 为空时 `hi = 1.0`。

In [ ]:
def feasible_nms_window(dup_ious, keep_ious):
    # TODO: 返回 (lo, hi, ok)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
lo, hi, ok = feasible_nms_window([0.78, 0.67, 0.59], [0.27, 0.31])
assert abs(lo - 0.31) < 1e-9 and abs(hi - 0.59) < 1e-9 and ok, (lo, hi, ok)
lo2, hi2, ok2 = feasible_nms_window([0.78, 0.67, 0.59], [0.27, 0.69])
assert not ok2, '0.69 > 0.59，窗口应为空'
lo3, hi3, ok3 = feasible_nms_window([0.9], [])
assert ok3 and abs(lo3 - 0.0) < 1e-12 and abs(hi3 - 0.9) < 1e-12
lo4, hi4, ok4 = feasible_nms_window([], [0.4])
assert ok4 and abs(hi4 - 1.0) < 1e-12
# 用本 notebook 真实场景的数据再验一次
lo5, hi5, ok5 = feasible_nms_window(list(dup_ious), [iou_pair])
assert not ok5, '组合标志牌场景下窗口必须是空的'
print(f'真实 TSR 场景窗口 = [{lo5:.4f}, {hi5:.4f})  非空? {ok5}')
print('✅ 练习 1 通过：**这个窗口是否非空，是判断「该不该上一对一匹配」的最直接依据**')

## ✏️ 练习 2：重复率统计

实现 `duplicates_per_gt(pred_boxes, pred_scores, gt_boxes, score_thr=0.5, iou_thr=0.5)`：
返回长度为 M 的整数数组，第 j 个元素是「分数 ≥ `score_thr` **且** 与 GT j 的 IoU ≥ `iou_thr`」的预测个数。

理想值全是 1：等于 0 是漏检，大于 1 是重复。**这个指标是诊断 DETR 训练是否收敛的第一个信号**
（模块 05 会把它做成训练监控项）。

In [ ]:
def duplicates_per_gt(pred_boxes, pred_scores, gt_boxes, score_thr=0.5, iou_thr=0.5):
    # TODO: 用 iou_matrix，返回 shape=(M,) 的整数数组
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
d0 = duplicates_per_gt(P_BOX, P_SCORE, GT)
assert list(d0) == [4, 4, 3], d0
d1 = duplicates_per_gt(P_BOX[keep], P_SCORE[keep], GT)
assert list(d1) == [1, 1, 1], d1
d2 = duplicates_per_gt(q_box[out], q_score[out], GT)
assert list(d2) == [1, 1, 1], d2
# 阈值收紧后，边缘重复框应当被排除
d3 = duplicates_per_gt(P_BOX, P_SCORE, GT, score_thr=0.5, iou_thr=0.75)
assert list(d3) == [2, 2, 2], d3
d4 = duplicates_per_gt(P_BOX, P_SCORE, GT, score_thr=0.9)
assert list(d4) == [1, 0, 1], d4      # 只有 GT0(0.92) 和 GT2(0.95) 有 >=0.9 的框
print(f'密集预测 NMS 前 {d0.tolist()} -> NMS 后 {d1.tolist()} | 集合预测 {d2.tolist()}（未做抑制）')
print('✅ 练习 2 通过：**「重复率」比 mAP 更早暴露训练问题** —— 它在第几个 epoch 降到 1，')
print('   直接反映一对一监督有没有生效。')

## ✏️ 练习 3：贪心匹配为什么不够

实现 `greedy_one_to_one(cost)`：反复取全局最小的未占用格子作为一组配对，
直到行或列用完。返回 `(rows, cols, total)`，`rows` 按升序排列。

然后对照 `brute_force_assignment` 的最优解——**贪心不是最优**，
这正是我们下一模块必须学匈牙利算法的原因。

In [ ]:
def greedy_one_to_one(cost):
    # TODO: 每次取剩余矩阵的全局最小，配对后划掉该行该列
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
TRAP = np.array([[1.0,   2.0],
                 [2.0, 100.0]])
gr, gc, gtot = greedy_one_to_one(TRAP)
_, _, otot = brute_force_assignment(TRAP)
assert abs(gtot - 101.0) < 1e-9, f'贪心先拿走 C[0,0]=1，剩下只能吃 100，总代价 101，得到 {gtot}'
assert abs(otot -   4.0) < 1e-9, '最优是 (0->1)+(1->0) = 2+2 = 4'
assert gtot > otot
print(f'贪心总代价 {gtot:.1f}  vs  最优总代价 {otot:.1f}   ->  贪心差了 {gtot/otot:.1f} 倍')

# 在本 notebook 的真实代价矩阵上贪心恰好等于最优（小矩阵常常如此，别被骗了）
gr2, gc2, gt2 = greedy_one_to_one(COST)
_, _, ot2 = brute_force_assignment(COST)
print(f'本节 6x2 代价矩阵：贪心 {gt2:.4f} / 最优 {ot2:.4f}  -> {"相同" if abs(gt2-ot2)<1e-9 else "不同"}')
# 随机大量抽样，统计贪心的次优比例
rng2 = np.random.default_rng(7); bad = 0; TRIALS = 300
for _ in range(TRIALS):
    A = np.round(rng2.normal(size=(5, 4)) * 3, 2)
    if greedy_one_to_one(A)[2] > brute_force_assignment(A)[2] + 1e-9:
        bad += 1
print(f'随机 5x4 代价矩阵 {TRIALS} 次：贪心给出次优解的比例 = {bad/TRIALS:.1%}')
assert bad / TRIALS > 0.05, '贪心应当在相当比例的随机矩阵上次优'
print('✅ 练习 3 通过：**贪心「局部最便宜」会锁死后面的选择** —— 必须用全局最优的匈牙利算法。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def feasible_nms_window(dup_ious, keep_ious):
    lo = float(max(keep_ious)) if len(keep_ious) else 0.0   # 保住真目标：thr >= max(keep)
    hi = float(min(dup_ious))  if len(dup_ious)  else 1.0   # 删光重复框：thr <  min(dup)
    return lo, hi, lo < hi

In [ ]:
# 练习 2 参考答案
def duplicates_per_gt(pred_boxes, pred_scores, gt_boxes, score_thr=0.5, iou_thr=0.5):
    if len(pred_boxes) == 0:
        return np.zeros(len(gt_boxes), dtype=int)
    M = iou_matrix(np.asarray(pred_boxes, float), np.asarray(gt_boxes, float))
    hit = (np.asarray(pred_scores, float)[:, None] >= score_thr) & (M >= iou_thr)
    return hit.sum(axis=0).astype(int)

In [ ]:
# 练习 3 参考答案
def greedy_one_to_one(cost):
    a = np.array(cost, dtype=float)
    n, m = a.shape
    rows, cols = [], []
    for _ in range(min(n, m)):
        i, j = np.unravel_index(int(np.argmin(a)), a.shape)
        rows.append(int(i)); cols.append(int(j))
        a[i, :] = np.inf; a[:, j] = np.inf          # 划掉该行该列
    order = np.argsort(rows)
    rows = np.array(rows)[order]; cols = np.array(cols)[order]
    total = float(np.asarray(cost, dtype=float)[rows, cols].sum())
    return rows, cols, total

---
## 🧪 真实工程胶囊：把「两种范式」写进配置与验收标准

In [ ]:
RECIPE = r'''
# ============================================================
# A. 密集检测器（YOLO / RTMDet）的后处理配置 —— 注意每一项都是**手工先验**
# ============================================================
model.test_cfg = dict(
    score_thr=0.05,          # 太低 -> NMS 输入框数暴涨 -> 延迟 p99 失控
    nms=dict(type='nms', iou_threshold=0.65),   # <-- 全局标量阈值，本课的靶心
    max_per_img=300,         # 截断保护：没有它，逆光/隧道口会把延迟拖爆
    nms_pre=1000,            # NMS 前先按分数取 top-k
)
# 车端验收必须额外测：**不同目标数下的后处理耗时分布**，而不只是平均 FPS
#   for n_obj in [1, 5, 20, 100, 500]: measure_p50_p99(nms_latency)

# ============================================================
# B. DETR 系的对应配置 —— 后处理只剩 top-k
# ============================================================
model.test_cfg = dict(max_per_img=100)      # 就这一项；**没有 nms 字段**
# 训练侧多出来的是 matcher：
train_cfg = dict(assigner=dict(
    type='HungarianAssigner',
    match_costs=[dict(type='ClassificationCost',  weight=1.0),   # 用**概率**不是 log
                 dict(type='BBoxL1Cost',          weight=5.0, box_format='xywh'),
                 dict(type='IoUCost', iou_mode='giou', weight=2.0)]))

# ============================================================
# C. 选型自检清单（把本 notebook 的量化结论变成可执行的检查）
# ============================================================
CHECKLIST = [
  ("统计数据集里「必须同时保留的重叠真目标对」的 IoU 分布",
   "取 95 分位数 -> 这是 NMS 阈值的下界 lo"),
  ("统计当前检测器输出的「同目标重复框对」IoU 分布",
   "取 5 分位数 -> 这是 NMS 阈值的上界 hi"),
  ("若 lo >= hi -> 单一 NMS 阈值不存在，**这是选 DETR 系的硬理由**", ""),
  ("实测后处理耗时随目标数的曲线，报告 p99/p50 比值", "> 2.0 说明延迟方差不可接受"),
  ("DETR 系上线仍建议挂一个 iou_thr=0.9 的兜底 NMS", "训练不充分时重复率不为零"),
]
for i, (k, v) in enumerate(CHECKLIST, 1):
    print(f"{i}. {k}")
    if v:
        print(f"     -> {v}")
'''
print(RECIPE)
for token in ['HungarianAssigner', 'iou_threshold', 'max_per_img',
              'ClassificationCost', 'giou', 'p99/p50', 'lo >= hi']:
    assert token in RECIPE, token
print('✅ 胶囊覆盖：密集检测器后处理配置 / DETR matcher 配置 / 选型量化自检清单')

### 小结

- **核心命题**：NMS 是在**推理期删掉**重复，一对一匹配是在**训练期不让**重复长出来。
  前者是打补丁（不可微、阈值全局、延迟数据依赖），后者是改目标函数。
- **NMS 的失败是范式问题而非实现问题**：它用一个全局标量阈值回答「这两个框是不是同一个目标」，
  而这个问题的答案依赖场景。本 notebook 在真实 TSR 组合标志牌场景上算出
  **可行阈值窗口 = [0.694, 0.592) = 空集** —— 没有任何阈值能同时删光重复且不误删真目标。
- **三个机制缺一不可**：① 二分图最优匹配建立对应关系；② 集合损失里的 no-object
  提供压制重复的力；③ object query 的 self-attention 提供去重的**能力**。
  匹配给动机，attention 给能力。
- **贪心匹配不够**：随机代价矩阵上有相当比例会给出次优解，锁死后面的选择。
  下一模块要手写 O(n³) 的匈牙利算法。
- **对 TSR 的真实价值**是「延迟确定性」与「密集重叠不误删」，**不是**纸面 AP。
  面试里说「DETR 没有 NMS 所以更快」是错的；正确说法是「延迟方差更小、p99 更接近 p50」。
- 原版 DETR 直接用在 TSR 上会很难看（单尺度 stride-32 特征 + 100 query）。
  **多尺度 + 可变形注意力是 TSR 落地 DETR 系的前提条件**，不是可选项。

下一站：**模块 01 · 二分图匹配与匈牙利算法** —— 本课最重要的一章，从零手写 O(n³) 匹配。